# Step 3: Handling the Imbalance

Step 2's baseline caught only 58.9% of real fraud - trained as if the two classes were roughly equal, when in reality fraud is 0.167% of the data. This step tries two standard techniques that explicitly tell the model "the minority class matters more than its tiny share of the data would suggest": **class weights** and **SMOTE**.

In [1]:
from data_prep import load_data, prepare_split

df = load_data()
X_train, X_test, y_train, y_test = prepare_split(df)

## Technique 1: Class weights

**What it does:** normally, a model's training process treats every mistake equally - getting one fraud case wrong "costs" the same as getting one legitimate case wrong. With `class_weight="balanced"`, mistakes on the minority class (fraud) get penalized much more heavily during training - roughly in proportion to how rare that class is. With fraud at ~0.17%, a false negative on a fraud case now costs the model far more than a false positive on a legitimate one, so it adjusts its decision boundary to catch more fraud, at the cost of flagging more legitimate transactions along the way. No new data is created - it's purely a change in how mistakes are scored during training.

In [2]:
from models import train_logistic_regression, evaluate

clf_weighted = train_logistic_regression(X_train, y_train, class_weight="balanced")
metrics_w = evaluate(clf_weighted, X_test, y_test)
y_pred_w, y_proba_w = metrics_w["y_pred"], metrics_w["y_proba"]

print(f"Precision: {metrics_w['precision']*100:.1f}%")
print(f"Recall:    {metrics_w['recall']*100:.1f}%")
print(f"F1:        {metrics_w['f1']:.3f}")
print(f"ROC-AUC:   {metrics_w['roc_auc']:.3f}")

Precision: 5.6%
Recall:    87.4%
F1:        0.106
ROC-AUC:   0.966


## Technique 2: SMOTE (Synthetic Minority Oversampling Technique)

**What it does:** instead of changing how mistakes are scored, SMOTE changes the training data itself - it creates brand new, *synthetic* fraud examples by picking a real fraud case, finding its nearest fraud neighbors (in the 30-feature space), and generating a new point somewhere between them. Do this enough times and the training set ends up with roughly equal numbers of fraud and legitimate examples.

**Critical rule: SMOTE only ever touches the training set, never the test set.** The test set has to reflect the real world's true ~0.17% fraud rate - evaluating against a test set padded with synthetic fraud examples would make the model look better than it actually is on real, unseen data. `prepare_split()` already separated train and test before this notebook touches SMOTE, so there's no risk of that leaking in here.

In [3]:
from models import apply_smote

print("Before SMOTE:", y_train.value_counts().to_dict())

X_train_smote, y_train_smote = apply_smote(X_train, y_train)

print("After SMOTE: ", y_train_smote.value_counts().to_dict())

Before SMOTE: {0: 226602, 1: 378}


After SMOTE:  {0: 226602, 1: 226602}


In [4]:
clf_smote = train_logistic_regression(X_train_smote, y_train_smote)
metrics_sm = evaluate(clf_smote, X_test, y_test)
y_pred_sm, y_proba_sm = metrics_sm["y_pred"], metrics_sm["y_proba"]

print(f"Precision: {metrics_sm['precision']*100:.1f}%")
print(f"Recall:    {metrics_sm['recall']*100:.1f}%")
print(f"F1:        {metrics_sm['f1']:.3f}")
print(f"ROC-AUC:   {metrics_sm['roc_auc']:.3f}")

Precision: 5.3%
Recall:    87.4%
F1:        0.100
ROC-AUC:   0.962


## Comparing all three, honestly

In [5]:
import pandas as pd

comparison = pd.DataFrame({
    "Baseline (Step 2)": [0.848, 0.589, 0.696, 0.956],
    "Class weights": [metrics_w["precision"], metrics_w["recall"], metrics_w["f1"], metrics_w["roc_auc"]],
    "SMOTE": [metrics_sm["precision"], metrics_sm["recall"], metrics_sm["f1"], metrics_sm["roc_auc"]],
}, index=["Precision", "Recall", "F1", "ROC-AUC"])

comparison.round(3)

,Baseline (Step 2),Class weights,SMOTE
Precision,0.848,0.056,0.053
Recall,0.589,0.874,0.874
F1,0.696,0.106,0.100
ROC-AUC,0.956,0.966,0.962


**This result needs to be read carefully, not just reported.** Both techniques did exactly what they promise: recall jumped from 58.9% to **87.4%** - the model now catches far more real fraud. But precision collapsed from 84.8% to roughly **5-6%** - meaning for every real fraud case flagged, there are now around 17 false alarms on legitimate transactions. F1 actually got *worse* (0.70 -> ~0.10), because F1 needs both precision and recall to be reasonable, and precision just cratered.

**Is this a failure?** Not exactly - and this is the most important thing to understand about imbalance handling: **ROC-AUC barely moved** (0.956 -> 0.966 / 0.962). That means the models' underlying ability to *rank* fraud above legitimate transactions didn't actually get much better or worse - what changed is where the *default* 50% probability cutoff lands relative to that ranking. Class weights and SMOTE both shifted the model's internal sense of "how likely is fraud" so far that the default threshold now flags almost anything even slightly suspicious. The real fix isn't necessarily choosing one of these techniques over the baseline - it's **not blindly trusting the default 0.5 threshold** for a problem this imbalanced. Step 5 tackles that directly.

For now: class weights and SMOTE reached near-identical results here, but class weights required no extra data manipulation and trained just as well - a good default choice going into Step 4.

## Summary

| Metric | Baseline | Class Weights | SMOTE |
|---|---|---|---|
| Precision | 84.8% | ~5.6% | ~5.3% |
| Recall | 58.9% | ~87.4% | ~87.4% |
| F1 | 0.696 | ~0.106 | ~0.100 |
| ROC-AUC | 0.956 | ~0.966 | ~0.962 |

- Both imbalance-handling techniques dramatically raise recall, at a steep cost to precision - a real trade-off, not a free win.
- Stable ROC-AUC across all three shows the *ranking* quality didn't change much - the *default classification threshold* is the real lever here, not just the training technique.
- Class weights and SMOTE perform almost identically in this case, with class weights being simpler (no synthetic data generation needed).

Next: Step 4 trains a fundamentally stronger model (Random Forest) to see if a better algorithm - not just better handling of the imbalance - can improve on this trade-off.